# Week 1 (interactive) — From your first API call to a weather bot 🤖🌦️

This notebook takes the tiny `hello_llm.py` from the Week 1 page and grows it, one small change at a time, into a chatbot that can **call tools** (run real Python functions) to answer questions it couldn't answer on its own.

You'll go through **6 steps**:

1. Change the **prompt** and the **model**
2. Take the prompt from **user input**
3. Send **several messages** between you and the LLM
4. Build a real **conversation loop** (it remembers, runs until you type `exit`)
5. The **simplest tool call** — give the model a calculator
6. A **real tool** — live weather with [`python-weather`](https://pypi.org/project/python-weather/)

> **This is the optional "jump start" from the onboarding email.** You need your NRP token to run it (steps 3 & 4 in the email). No token yet? Do the Week 2 Python notebook first — it needs no token.

## Before you run anything

1. You have a `.env` file next to this notebook with:
   ```
   NRP_LLM_TOKEN=sk-...your token...
   NRP_LLM_BASE_URL=https://ellm.nrp-nautilus.io/v1
   ```
   ⚠️ **Never** paste your token into a cell, a screenshot, or git. It lives in `.env` only.
2. Run the install cell once (next cell).
3. Then run the setup cell. After that, run the steps in order — each one builds on the last.

**How to run a cell:** click it and press **Shift+Enter**.

In [ ]:
# Run this ONCE to install the libraries this notebook uses.
# (The leading ! runs a terminal command from inside the notebook.)
!pip install openai python-dotenv python-weather nest_asyncio

In [ ]:
# Setup — run this every time you open the notebook.
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads your .env file

client = OpenAI(
    api_key=os.environ["NRP_LLM_TOKEN"],
    base_url=os.environ["NRP_LLM_BASE_URL"],
)

MODEL = "gpt-oss"   # we'll change this later
print("Ready. Using model:", MODEL)

## Step 0 — the baseline (this is `hello_llm.py`)

This is exactly the program from the Week 1 page: one question in, one answer out. Run it.

In [ ]:
response = client.chat.completions.create(
    model="gpt-oss",
    messages=[{"role": "user", "content": "What is the National Research Platform?"}],
)
print(response.choices[0].message.content)

## Step 1 — change the prompt, the model, and the settings 🔧

Every call has things you control. The two obvious ones:
- **`content`** — what you ask (the *prompt*)
- **`model`** — which AI brain answers

But you also get **settings** ("parameters") that change *how* the model answers:

| Parameter | What it does | Try |
|---|---|---|
| **`temperature`** | Creativity / randomness. `0.0` = focused & repeatable, higher = wilder. | `0.0` to `1.5` |
| **`max_tokens`** | The longest the answer can be (a token ≈ ¾ of a word). Cuts it off if hit. | `30`, `200` |
| **`top_p`** | Another creativity knob (keeps only the most likely words). Usually leave at `1.0`. | `0.1` to `1.0` |
| **`presence_penalty`** | Pushes the model to bring up *new* topics. | `-2.0` to `2.0` |
| **`frequency_penalty`** | Discourages repeating the same words. | `-2.0` to `2.0` |
| **`seed`** | A fixed number → same answer every run (with `temperature=0`). Great for testing. | any int |

Let's pull the prompt and model into variables and pass some settings.

In [ ]:
PROMPT = "Explain what a GPU is to a 10-year-old, in 3 sentences."
MODEL  = "minimax-m2"   # a big model in NRP's "evaluating" tier; "gpt-oss" is the safe default

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": PROMPT}],
    temperature=0.7,          # 0 = focused/repeatable, higher = more creative
    max_tokens=200,           # cap the answer length
    top_p=1.0,                # nucleus sampling (creativity knob)
    presence_penalty=0.0,     # >0 nudges it toward new topics
    frequency_penalty=0.0,    # >0 discourages repeating words
    seed=42,                  # same seed + temperature 0 => same answer every time
)
print(response.choices[0].message.content)
print("\n--- tokens used:", response.usage.total_tokens,
      "(prompt", response.usage.prompt_tokens, "+ answer", response.usage.completion_tokens, ")")

### See `temperature` with your own eyes 🌡️

Same prompt, three temperatures. Low = safe and similar each run; high = creative and unpredictable. Run it a couple of times and watch the high-temperature one change the most.

In [ ]:
PROMPT = "Invent a name for a pet robot and describe it in one sentence."

for temp in [0.0, 0.8, 1.5]:
    print("=" * 50, "temperature =", temp)
    response = client.chat.completions.create(
        model="minimax-m2",
        messages=[{"role": "user", "content": PROMPT}],
        temperature=temp,
        max_tokens=60,
    )
    print(response.choices[0].message.content, "\n")

In [ ]:
# 🔧 Your turn: same prompt + same settings, four different models. Notice how they differ.
PROMPT = "Tell me a joke about supercomputers."

for model_name in ["gpt-oss", "qwen3-small", "gemma-small", "minimax-m2"]:
    print("=" * 50)
    print("MODEL:", model_name)
    print("=" * 50)
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=0.9,
        max_tokens=120,
    )
    print(response.choices[0].message.content)
    print()

## Step 2 — take the prompt from the user ⌨️

Hard-coding the question is boring. Let's ask the *person* what to send.

`input(...)` pops up a box, waits for you to type, and hands back what you typed as a string.

> **Note:** this cell pauses and waits for you. Type your question, press Enter.

In [ ]:
question = input("Ask the AI anything: ")

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
)
print()
print(response.choices[0].message.content)

## Step 3 — more messages between you and the LLM 💬

`messages` is a **list**, not a single string. Each item has a **role**:

- `system` — the bot's instructions / personality (optional, comes first)
- `user`   — something *you* said
- `assistant` — something the *bot* said

By writing out several messages yourself, you can set a personality **and** show the bot how the conversation has gone so far. Watch how the `system` message changes the whole tone.

In [ ]:
messages = [
    {"role": "system",    "content": "You are a pirate. Answer everything like a pirate, briefly."},
    {"role": "user",      "content": "What is a supercomputer?"},
    {"role": "assistant", "content": "Arr! A mighty ship o' many computers lashed together, sailin' as one!"},
    {"role": "user",      "content": "And what is a GPU?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

👆 Notice the bot answered the **second** question *in character* and *in context* — because we handed it the whole conversation, not just the latest line.

That's the key idea behind memory: **the model has no memory of its own.** It only knows what's in the `messages` list you send. If you want it to remember, *you* have to keep the list and keep adding to it. That's exactly Step 4.

## Step 4 — a real conversation (it remembers, until you type `exit`) 🔁

Now we put it in a loop:

1. Start a `messages` list (optionally with a `system` personality).
2. Ask the user for input.
3. Append their line as a `user` message.
4. Send the **whole list** and get a reply.
5. Append the reply as an `assistant` message (← this is the "memory").
6. Repeat until they type `exit`.

> **Note:** this cell runs a loop and keeps asking. Type `exit` (or `quit`) to stop it.

In [ ]:
messages = [
    {"role": "system", "content": "You are a friendly, concise tutor for high-school students."}
]

print("Chat started. Type 'exit' to quit.\n")
while True:
    user_input = input("You: ")
    if user_input.lower() in ("exit", "quit"):
        print("Bye!")
        break

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(model=MODEL, messages=messages)
    reply = response.choices[0].message.content

    messages.append({"role": "assistant", "content": reply})  # <-- remembers
    print("Bot:", reply, "\n")

👆 Try it: tell it your name, then a few messages later ask *"what's my name?"* It remembers — because every reply got appended back into `messages`.

This is `chat.py`, basically — the thing you build with your pair in **Week 2**. You just wrote it.

## Step 5 — the simplest tool call 🧮

Here's a fun fact: **LLMs are bad at arithmetic.** Ask one for `89231 * 47816` and it'll confidently make something up.

The fix: give the model a **tool** — a real Python function it can ask us to run. The dance:

1. We describe our function to the model (a "tool").
2. The model, instead of answering, says *"please run `multiply(89231, 47816)`."*
3. **We** run the real Python function and hand back the result.
4. The model uses that result to write the final answer.

Let's wire up one tool: `multiply`.

In [ ]:
# 1) The real Python function.
def multiply(a, b):
    return a * b

# 2) Describe it to the model. This JSON is the model's "menu" of tools.
tools = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two numbers together and return the exact result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "the first number"},
                    "b": {"type": "number", "description": "the second number"},
                },
                "required": ["a", "b"],
            },
        },
    }
]
print("Tool defined.")

In [ ]:
# 3) Ask a math question and let the model reach for the tool.
messages = [{"role": "user", "content": "What is 89231 * 47816? Use your tool."}]

response = client.chat.completions.create(model="gpt-oss", messages=messages, tools=tools)
msg = response.choices[0].message

if msg.tool_calls:
    messages.append(msg)  # record the model's request to call a tool
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)   # e.g. {"a": 89231, "b": 47816}
        result = multiply(**args)                    # WE run the real function
        print("Model asked for multiply(%s) -> %s" % (args, result))
        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": str(result),
        })
    # 4) Send the result back so the model can write the final sentence.
    final = client.chat.completions.create(model="gpt-oss", messages=messages, tools=tools)
    print("\nFinal answer:", final.choices[0].message.content)
else:
    print("(The model answered without the tool:)", msg.content)

That's the **entire** idea behind tools (and behind "AI agents"): the model decides *when* to call a function, you run it, you give back the result. Everything fancy is just more tools.

## Step 6 — a real tool: live weather 🌦️

A calculator is cute, but the model could *almost* do that itself. The magic shows when a tool does something the model **cannot** know — like today's weather.

We'll use [`python-weather`](https://pypi.org/project/python-weather/). One wrinkle: it's **async** (it uses `await`). To call it from our normal code, we wrap it. The `nest_asyncio` line lets async run smoothly inside a notebook.

In [ ]:
import asyncio, nest_asyncio
import python_weather

nest_asyncio.apply()  # lets us call async code from inside the notebook

async def _fetch_weather(city):
    async with python_weather.Client(unit=python_weather.IMPERIAL) as wclient:
        w = await wclient.get(city)
        return {
            "city": city,
            "temperature": w.temperature,
            "feels_like": w.feels_like,
            "description": w.description,
            "humidity": w.humidity,
            "wind_speed": w.wind_speed,
        }

def get_weather(city):
    # Return current weather for a city as a plain dict (a normal sync function).
    return asyncio.run(_fetch_weather(city))

# Test it directly — no LLM yet:
get_weather("San Diego")

> **Windows note:** if that cell errors with something about an event loop, add this line *above* `nest_asyncio.apply()` and re-run:
> ```python
> asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
> ```

Now add `get_weather` to our tool menu, alongside `multiply`, and write one loop that handles **any** number of tool calls.

In [ ]:
# Add the weather tool to the menu (keep multiply too).
tools = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two numbers and return the exact result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"},
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city anywhere in the world.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "the city name, e.g. 'Tokyo'"},
                },
                "required": ["city"],
            },
        },
    },
]

# A lookup from tool name -> the real Python function.
available = {"multiply": multiply, "get_weather": get_weather}
print("Tools ready:", list(available))

In [ ]:
def chat_with_tools(user_message):
    # Send a message; let the model call tools as many times as it needs.
    messages = [{"role": "user", "content": user_message}]
    while True:
        response = client.chat.completions.create(
            model="gpt-oss", messages=messages, tools=tools
        )
        msg = response.choices[0].message

        if not msg.tool_calls:        # model is done — return its answer
            return msg.content

        messages.append(msg)
        for call in msg.tool_calls:
            fn = available[call.function.name]
            args = json.loads(call.function.arguments)
            result = fn(**args)
            print("  [called %s(%s) -> %s]" % (call.function.name, args, result))
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })

# Try it!
print(chat_with_tools("What's the weather in Tokyo right now?"))
print()
print(chat_with_tools("If it's 25 degrees in 14 cities, what's 25 times 14? And how's the weather in Paris?"))

## 🎉 You did it

You started with a one-line question-and-answer and ended with a bot that **decides on its own** when to fetch live weather or do exact math, then weaves the results into a normal sentence. That's a real AI **agent**.

### Challenges (pick any)
- Add a `get_time(timezone)` tool so it can answer "what time is it in London?"
- Make the weather tool return the 3-day forecast (loop over `for daily in w:` inside `_fetch_weather`).
- Put `chat_with_tools` inside the Step 4 conversation loop so you can have a back-and-forth chat *with* tools.
- Give it a `system` personality and see if it still uses tools correctly.

This is exactly the **Week 8 bonus** (a Matrix bot + tool calling) — you're way ahead.